In [1]:
import numpy as np
from pyscf import gto, scf, lo, cc
import os

basis = 'ccpvdz'
atoms = '''
C    -1.0478252   -1.4216736    0.0000000
C    -1.4545034   -0.8554459    1.2062048
C    -1.4545034   -0.8554459   -1.2062048
C    -2.2667970    0.2771610    1.2069539
C    -2.6714781    0.8450211    0.0000000
C    -2.2667970    0.2771610   -1.2069539
H    -1.1338534   -1.2920593   -2.1423150
H    -2.5824943    0.7163066   -2.1437977
H    -3.3030422    1.7232700    0.0000000
H    -2.5824943    0.7163066    2.1437977
H    -1.1338534   -1.2920593    2.1423150
H    -0.4060253   -2.2919049    0.0000000
C     1.0478252    1.4216736    0.0000000
C     1.4545034    0.8554459   -1.2062048
C     1.4545034    0.8554459    1.2062048
C     2.2667970   -0.2771610   -1.2069539
C     2.6714781   -0.8450211    0.0000000
C     2.2667970   -0.2771610    1.2069539
H     0.4060253    2.2919049    0.0000000
H     1.1338534    1.2920593    2.1423150
H     2.5824943   -0.7163066    2.1437977
H     3.3030422   -1.7232700    0.0000000
H     2.5824943   -0.7163066   -2.1437977
H     1.1338534    1.2920593   -2.1423150
'''

mol = gto.M(atom = atoms,
            basis = basis,
            verbose = 4,
            unit = 'A',
            symmetry = 0,
            charge = 0,
            spin = 0,
            max_memory = 200000,
            )

mf = scf.RHF(mol).density_fit()
mf.max_cycle = 100
mf.level_shift = 0.5
mf.kernel()

stable = False
for i in range(10):
    print(f'mf stability test {i+1}')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'mf energy: {mf.e_tot}, stability {stable}')
        break


System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Sat Aug 15 22:36:38 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 24
[INPUT] num. electrons = 84
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      uni

In [2]:
umf = mf.to_uhf()

In [3]:
from pyscf.data import elements
import lno_tools
from pyscf.lno import lnoccsd, ulnoccsd, tools
from pyscf.lno.tools import autofrag_iao

iao_coeff, iao_frag_list, atm_center = lno_tools.iao_localization(mf)

In [4]:
from afqmc.lno_afqmc import lno_afqmc
from afqmc.lno_afqmc import tools as lnoafqmc_tools
from jax import random

def run_lno(mf,
            lo_coeff = None, 
            frag_lolist = None,
            nfrozen = 0,
            lno_thresh = 1e-6,
            qmc_options = {}, 
            chol_cut = 1e-5, 
            target_sto_error = 1e-3, 
            run_frag_list = None, 
            atom_group = None,
            ):

    print("\n ******* LNO-CALCULATION ******* \n")

    lno_tools.check_span(mf, lo_coeff, nfrozen, thresh=1e-10)

    if isinstance(mf, scf.rhf.RHF):
        spin_type = "restricted"
        mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=mf.verbose)
    elif isinstance(mf, scf.uhf.UHF):
        spin_type = "unrestricted"
        mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=mf.verbose)
    else:
        raise TypeError(f'unsupported mean-field type: {type(mf)}')

    if isinstance(lno_thresh, float):
        mlno.lno_thresh = [lno_thresh*10, lno_thresh]
    elif isinstance(lno_thresh, (list, tuple)):
        assert len(lno_thresh) == 2
        mlno.lno_thresh = [lno_thresh[0], lno_thresh[1]]

    lno_thresh = mlno.lno_thresh
    print(f"LNO THRESHOLD = {mlno.lno_thresh}")
    lno_type = ['1h','1h']
    eris = mlno.ao2mo()

    nfrag_tot = len(frag_lolist)
    if run_frag_list is None:
        run_frag_list = range(nfrag_tot)

    frag_lolist = [frag_lolist[i] for i in run_frag_list]
    nfrag_run = len(frag_lolist)

    lno_pct_occ = [None, None]
    lno_norb = [[None,None]] * nfrag_tot

    las_center = [None]*nfrag_run
    las_size = [None]*nfrag_run
    lno_emp = np.zeros(nfrag_run, dtype='float64')
    lno_ecc  = np.zeros(nfrag_run, dtype='float64')
    lno_eqmc  = np.zeros(nfrag_run, dtype='float64')
    lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')

    mol = mf.mol

    seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                           shape=(nfrag_tot,), 
                           minval=0, 
                           maxval=100*nfrag_tot
                           )
    
    qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)

    # Loop over fragment
    for ifrag, frag_idx in enumerate(run_frag_list):
        
        loidx = frag_lolist[ifrag]

        print("\n")
        width = 80
        msg = f" {spin_type} LNO-FRAGMENT {frag_idx+1}/({nfrag_run},{nfrag_tot}) "
        print(msg.center(width, '='))
        if atom_group is not None:
            loc_ctr = f"{atom_group[frag_idx]}"
            print(f"Center Atom {loc_ctr}")
        else:
            loc_ctr = None

        orbloc, lno_param \
            = lno_tools.get_lnoparam(mlno, lo_coeff, lno_thresh, lno_pct_occ, lno_norb, loidx, ifrag)

        lno_coeff, lno_frozen, uocc_loc, _ \
                    = mlno.make_las(eris, orbloc, lno_type, lno_param)
        
        if isinstance(mlno._scf, scf.rhf.RHF):
            lno_frozen, maskact \
                = lnoccsd.get_maskact(lno_frozen, mlno.mo_occ.size)
        elif isinstance(mlno._scf, scf.uhf.UHF):
            lno_frozen, maskact \
                = ulnoccsd.get_maskact(lno_frozen, [mlno.mo_occ[0].size, mlno.mo_occ[1].size])
        else:
            raise TypeError(f'unsupported mean-field type: {type(mlno._scf)}')

        _, nlno = lnoafqmc_tools.split_lno(mlno, lno_coeff, lno_frozen)

        eorb_mp = lno_tools.lnomp2_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)

        eorb_cc, t1, t2 = \
            lno_tools.lnoccsd_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)

        eorb_qmc, eorb_qmc_err \
            = lno_afqmc.lnoafqmc_kernel(
                mlno, lno_coeff, uocc_loc, lno_frozen, t1, t2, 
                chol_cut, frag_idx, seeds, qmc_options)

        print(f'LNO-Active Space Size     {nlno}')
        print(f'LNO-MP2 Orbital Energy    {eorb_mp:.8f}')
        print(f'LNO-CCSD Orbital Energy   {eorb_cc:.8f}')
        print(fr'LNO-AFQMC Orbital Energy  {eorb_qmc:.5f} {eorb_qmc_err:.5f}')

        las_size[ifrag] = nlno
        lno_emp[ifrag] = eorb_mp
        lno_ecc[ifrag] = eorb_cc
        lno_eqmc[ifrag] = eorb_qmc
        lno_eqmc_err[ifrag] = eorb_qmc_err

        qmc_err = np.sqrt(np.sum(np.array(lno_eqmc_err)**2))

    return np.array(las_size).max(), sum(lno_emp), sum(lno_ecc), sum(lno_eqmc), qmc_err

In [8]:
options = {
           'eql_time': 10,
           'n_prop_steps': 50,
           'n_blocks': 100,
           'n_walkers': 100,
           'mix_precision': False,
           'seed': 17,
           'walker_type': 'rhf',
           'trial': 'pt2ccsd',
           }

threshs = [3e-5]
las = np.zeros(len(threshs))
elnomp = np.zeros(len(threshs))
elnocc = np.zeros(len(threshs))
elnoqmc = np.zeros(len(threshs))
elnoqmc_err = np.zeros(len(threshs))

for i, lno_thresh in enumerate(threshs):
    las[i], elnomp[i], elnocc[i], elnoqmc[i], elnoqmc_err[i] = \
        run_lno(mf,
                lo_coeff = iao_coeff,
                frag_lolist = iao_frag_list,
                nfrozen = elements.chemcore(mol),
                lno_thresh = 3e-5,
                qmc_options = options,
                chol_cut = 1e-4,
                target_sto_error = 1e-4,
                run_frag_list = [23],
                atom_group = atm_center)


 ******* LNO-CALCULATION ******* 

LO span the MO occ space - True.
MO occ span the LO space - False.
LNO THRESHOLD = [0.00030000000000000003, 3e-05]


====================== restricted LNO-FRAGMENT 24/(1,24) =======================
Center Atom H
LO occ proj: 1 active | 0 standby | 29 orthogonal
LO vir proj: 1 active | 0 standby | 185 orthogonal
nfrozen occupied orbitals:  35
nactive occupied orbitals:  7
nactive virtual orbitals:   29
nfrozen virtual orbitals:   157
Init t2, MP2 energy = -461.679799958277  E_corr(MP2) -0.242900760300484

******** <class 'pyscf.lno.lnoccsd.MODIFIED_CCSD'> ********
CC2 = 0
CCSD nocc = 7, nmo = 36
frozen orbitals [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 1

In [10]:
uiao_coeff, uiao_frag_list, atm_center = lno_tools.iao_localization(umf)

options = {
           'eql_time': 10,
           'n_prop_steps': 50,
           'n_blocks': 100,
           'n_walkers': 100,
           'mix_precision': False,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

threshs = [3e-5]
las = np.zeros(len(threshs))
elnomp = np.zeros(len(threshs))
elnocc = np.zeros(len(threshs))
elnoqmc = np.zeros(len(threshs))
elnoqmc_err = np.zeros(len(threshs))

for i, lno_thresh in enumerate(threshs):
    las[i], elnomp[i], elnocc[i], elnoqmc[i], elnoqmc_err[i] = \
        run_lno(umf,
                lo_coeff = uiao_coeff,
                frag_lolist = uiao_frag_list,
                nfrozen = elements.chemcore(mol),
                lno_thresh = 3e-5,
                qmc_options = options,
                chol_cut = 1e-4,
                target_sto_error = 1e-4,
                run_frag_list = [23],
                atom_group = atm_center)


 ******* LNO-CALCULATION ******* 

LO span the MO occ space - True.
MO occ span the LO space - False.
LNO THRESHOLD = [0.00030000000000000003, 3e-05]


===================== unrestricted LNO-FRAGMENT 24/(1,24) ======================
Center Atom H
LO occ proj: 1 active | 0 standby | 29 orthogonal
LO occ proj: 1 active | 0 standby | 29 orthogonal
nfrozen occupied orbitals:  [35, 35]
nactive occupied orbitals:  [7, 7]
nactive virtual orbitals:   [29, 29]
nfrozen virtual orbitals:   [157, 157]

WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations

Init t2, MP2 energy = -0.242693840665822

WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations


******** <class 'pyscf.lno.ulnoccsd.MODIFIED_UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(7), np.int64(7